import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from src.train_utils import train_one_epoch
from src.eval_utils import evaluate, evaluate_detailed, get_all_probas_and_labels, compute_roc_auc_scores, compute_pr_auc_scores

OUT_DIR = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'outputs/error_analysis/MLP/phase1_diagnostics')
os.makedirs(OUT_DIR, exist_ok=True)

device = ('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'OUT_DIR: {OUT_DIR}')


In [1]:
import sys; sys.path.append('../../..')   
import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from src.train_utils import train_one_epoch
from src.eval_utils import evaluate, evaluate_detailed, get_all_probas_and_labels, compute_roc_auc_scores, compute_pr_auc_scores

OUT_DIR = "../outputs/error_analysis/MLP/phase1_diagnostics"
os.makedirs(OUT_DIR, exist_ok=True)

device = ('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Dataset

In [2]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_ds = __import__('torchvision').datasets.FashionMNIST(root='../data', train=True, download=True, transform=transform)
test_ds = __import__('torchvision').datasets.FashionMNIST(root='../data', train=False, download=True, transform=transform)
class_names = train_ds.classes

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(f'Train: {len(train_loader)} batches  Test: {len(test_loader)} batches')

100%|██████████| 26.4M/26.4M [00:04<00:00, 5.68MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 131kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 1.78MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


Train: 938 batches  Test: 40 batches


## MLP Architecture — 784→256→128→10, ReLU, Dropout 0.2

In [3]:
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_1=256, hidden_2=128, num_classes=10, dropout=0.2):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_1)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_1, hidden_2)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_2, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x); x = self.relu1(x); x = self.drop1(x)
        x = self.fc2(x); x = self.relu2(x); x = self.drop2(x)
        return self.fc3(x)

model = MLP().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

Params: 235,146


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## A1 — Baseline Training (10 epochs, matching practice_1)

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses_10 = []
model.train()
for epoch in range(10):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses_10.append(loss)
    print(f'Epoch [{epoch+1}/10] Loss: {loss:.4f}')

with open(os.path.join(OUT_DIR, 'train_losses_10.txt'), 'w') as f:
    for l in train_losses_10: f.write(f'{l}\n')

Epoch [1/10] Loss: 0.5409
Epoch [2/10] Loss: 0.4101
Epoch [3/10] Loss: 0.3785
Epoch [4/10] Loss: 0.3545
Epoch [5/10] Loss: 0.3382
Epoch [6/10] Loss: 0.3238
Epoch [7/10] Loss: 0.3131
Epoch [8/10] Loss: 0.3022
Epoch [9/10] Loss: 0.2956
Epoch [10/10] Loss: 0.2868


## A1 — Baseline Evaluation

In [5]:
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name='MLP_baseline')
cm_np = cm.cpu().numpy()

with open(os.path.join(OUT_DIR, 'metrics_summary_10.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'\n{"Class":<15} {"TPR":>8} {"Precision":>10}\n')
    f.write('-' * 33 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']; prec = per_class[name]['Precision']
        f.write(f'{name:<15} {tpr:>8.4f} {prec:>10.4f}\n')

with open(os.path.join(OUT_DIR, 'confusion_matrix_10.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names: f.write(f'{name:>15}')
    f.write('\n')
    for i in range(10):
        f.write(f'{class_names[i]:>15}')
        for j in range(10): f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis_10.txt'), 'w') as f:
    f.write('Baseline (10 epochs) Misclassification\n' + '=' * 70 + '\n\n')
    for c in range(10):
        name = class_names[c]; errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {name} (errors: {errors})\n' + '-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0: continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f'Baseline: Acc={accuracy:.2f}%')
print(f'Per-class TPR:')
for name in class_names:
    print(f'  {name:<15} TPR={per_class[name]["TPR"]:.4f} Prec={per_class[name]["Precision"]:.4f}')

  Test Accuracy: 88.53%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8680     0.0230     0.8074
  Trouser             0.9650     0.0010     0.9908
  Pullover            0.7790     0.0160     0.8440
  Dress               0.8960     0.0149     0.8699
  Coat                0.8450     0.0251     0.7890
  Sandal              0.9440     0.0026     0.9762
  Shirt               0.6760     0.0271     0.7348
  Sneaker             0.9550     0.0087     0.9245
  Bag                 0.9700     0.0033     0.9700
  Ankle boot          0.9550     0.0058     0.9484
Baseline: Acc=88.53%
Per-class TPR:
  T-shirt/top     TPR=0.8680 Prec=0.8074
  Trouser         TPR=0.9650 Prec=0.9908
  Pullover        TPR=0.7790 Prec=0.8440
  Dress           TPR=0.8960 Prec=0.8699
  Coat            TPR=0.8450 Prec=0.7890
  Sandal          TPR=0.9440 Prec=0.9762
  Shirt           TPR=0.6760 Prec=0.7348
  Sneaker         TPR=0.9550 Prec=0.9245
  

## A2 — Extended Training (30 epochs + CosineAnnealingLR)

In [6]:
model_ext = MLP().to(device)
optimizer_ext = optim.Adam(model_ext.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer_ext, T_max=30)
EPOCHS = 30

train_losses_30 = []
model_ext.train()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model_ext, train_loader, criterion, optimizer_ext, device)
    train_losses_30.append(loss)
    scheduler.step()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}')

torch.save(model_ext.state_dict(), os.path.join(OUT_DIR, 'model_weights.pth'))
with open(os.path.join(OUT_DIR, 'train_losses_30.txt'), 'w') as f:
    for l in train_losses_30: f.write(f'{l}\n')
print(f'Done. Final loss: {train_losses_30[-1]:.4f}')

Epoch [1/30] Loss: 0.5410
Epoch [5/30] Loss: 0.3327
Epoch [10/30] Loss: 0.2755
Epoch [15/30] Loss: 0.2291
Epoch [20/30] Loss: 0.1865
Epoch [25/30] Loss: 0.1602
Epoch [30/30] Loss: 0.1480
Done. Final loss: 0.1480


## A2 — Extended Model Evaluation

In [7]:
accuracy_e, cm_e, per_class_e = evaluate_detailed(model_ext, test_loader, device, class_names, model_name='MLP_extended')
cm_np_e = cm_e.cpu().numpy()

with open(os.path.join(OUT_DIR, 'metrics_summary_30.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy_e:.2f}\n')
    f.write(f'\n{"Class":<15} {"TPR":>8} {"Precision":>10}\n')
    f.write('-' * 33 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class_e[name]['TPR']; prec = per_class_e[name]['Precision']
        f.write(f'{name:<15} {tpr:>8.4f} {prec:>10.4f}\n')

with open(os.path.join(OUT_DIR, 'confusion_matrix_30.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names: f.write(f'{name:>15}')
    f.write('\n')
    for i in range(10):
        f.write(f'{class_names[i]:>15}')
        for j in range(10): f.write(f'{cm_np_e[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis_30.txt'), 'w') as f:
    f.write('Extended (30 epochs) Misclassification\n' + '=' * 70 + '\n\n')
    for c in range(10):
        name = class_names[c]; errors = cm_np_e[c].sum() - cm_np_e[c, c]
        f.write(f'True: {name} (errors: {errors})\n' + '-' * 50 + '\n')
        for p in np.argsort(-cm_np_e[c]):
            if p == c or cm_np_e[c, p] == 0: continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np_e[c, p]:>4}\n')
        f.write('\n')

print(f'Extended: Acc={accuracy_e:.2f}%')

  Test Accuracy: 89.78%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8540     0.0174     0.8447
  Trouser             0.9730     0.0009     0.9918
  Pullover            0.8380     0.0213     0.8136
  Dress               0.9120     0.0119     0.8950
  Coat                0.8420     0.0184     0.8353
  Sandal              0.9600     0.0032     0.9707
  Shirt               0.7030     0.0272     0.7416
  Sneaker             0.9620     0.0063     0.9441
  Bag                 0.9700     0.0029     0.9739
  Ankle boot          0.9640     0.0039     0.9650
Extended: Acc=89.78%


## A3 — Logit Bias Sweep (zero-cost on extended model)

In [8]:
SHIRT_IDX = 6  # class index for Shirt
BIASES = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
sweep = []

model_ext.eval()
with torch.no_grad():
    for bias in BIASES:
        all_p, all_l = [], []
        sh_tp = sh_fp = sh_fn = 0
        for inputs, lbls in test_loader:
            inputs, lbls = inputs.to(device), lbls.to(device)
            logits = model_ext(inputs)
            logits[:, SHIRT_IDX] += bias
            preds = logits.argmax(dim=1)
            all_p.extend(preds.cpu().numpy()); all_l.extend(lbls.cpu().numpy())
            for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
                if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp += 1
                if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp += 1
                if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn += 1
        from sklearn.metrics import accuracy_score
        acc = accuracy_score(all_l, all_p)
        sweep.append({'bias': bias, 'acc': round(acc*100, 2), 'tpr': round(sh_tp/(sh_tp+sh_fn+1e-8), 4), 'prec': round(sh_tp/(sh_tp+sh_fp+1e-8), 4)})
        print(f'bias={bias:+.1f}  acc={acc:.2f}%  Shirt TPR={sh_tp/(sh_tp+sh_fn+1e-8):.4f}  Prec={sh_tp/(sh_tp+sh_fp+1e-8):.4f}')

bt = max(sweep, key=lambda r: r['acc'] + r['tpr'] * 100)
print(f'\nBest trade-off: bias={bt["bias"]:+.1f}  acc={bt["acc"]:.2f}%  TPR={bt["tpr"]:.4f}  Prec={bt["prec"]:.4f}')

with open(os.path.join(OUT_DIR, 'bias_sweep_results.txt'), 'w') as f:
    f.write(f'{"Bias":>6} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}\n' + '-' * 32 + '\n')
    for r in sweep:
        f.write(f'{r["bias"]:>+5.1f} {r["acc"]:>7.2f} {r["tpr"]:>9.4f} {r["prec"]:>10.4f}\n')
    f.write(f'\nBest trade-off: bias={bt["bias"]:+.1f}  acc={bt["acc"]:.2f}%  TPR={bt["tpr"]:.4f}  Prec={bt["prec"]:.4f}\n')

bias=-1.0  acc=0.89%  Shirt TPR=0.5730  Prec=0.8341
bias=-0.5  acc=0.90%  Shirt TPR=0.6480  Prec=0.7961
bias=+0.0  acc=0.90%  Shirt TPR=0.7030  Prec=0.7416
bias=+0.5  acc=0.90%  Shirt TPR=0.7650  Prec=0.6942
bias=+1.0  acc=0.89%  Shirt TPR=0.8140  Prec=0.6522
bias=+1.5  acc=0.88%  Shirt TPR=0.8540  Prec=0.5918
bias=+2.0  acc=0.88%  Shirt TPR=0.8810  Prec=0.5527

Best trade-off: bias=+2.0  acc=87.81%  TPR=0.8810  Prec=0.5527


## Comparison: Baseline (10) vs Extended (30) vs Bias Sweep

In [9]:
# Find worst class for MLP (not necessarily Shirt — different from CNN)
print(f'{"Config":<30} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}')
print('-' * 55)
print(f'{"Baseline 10 epochs":<30} {accuracy:>7.2f} {per_class["Shirt"]["TPR"]:>9.4f} {per_class["Shirt"]["Precision"]:>10.4f}')
print(f'{"Extended 30 epochs":<30} {accuracy_e:>7.2f} {per_class_e["Shirt"]["TPR"]:>9.4f} {per_class_e["Shirt"]["Precision"]:>10.4f}')
print(f'{"Extended + bias="+str(bt["bias"])+" (best)":<30} {bt["acc"]:>7.2f} {bt["tpr"]:>9.4f} {bt["prec"]:>10.4f}')

# Identify worst class for MLP
print(f'\nWorst TPR (baseline):')
sorted_tpr = sorted(class_names, key=lambda n: per_class[n]['TPR'])
for i, name in enumerate(sorted_tpr[:3]):
    print(f'  {i+1}. {name:<15} TPR={per_class[name]["TPR"]:.4f}  Prec={per_class[name]["Precision"]:.4f}')

print(f'\nWorst TPR (extended):')
sorted_tpr_e = sorted(class_names, key=lambda n: per_class_e[n]['TPR'])
for i, name in enumerate(sorted_tpr_e[:3]):
    print(f'  {i+1}. {name:<15} TPR={per_class_e[name]["TPR"]:.4f}  Prec={per_class_e[name]["Precision"]:.4f}')

Config                            Acc%  ShirtTPR  ShirtPrec
-------------------------------------------------------
Baseline 10 epochs               88.53    0.6760     0.7348
Extended 30 epochs               89.78    0.7030     0.7416
Extended + bias=2.0 (best)       87.81    0.8810     0.5527

Worst TPR (baseline):
  1. Shirt           TPR=0.6760  Prec=0.7348
  2. Pullover        TPR=0.7790  Prec=0.8440
  3. Coat            TPR=0.8450  Prec=0.7890

Worst TPR (extended):
  1. Shirt           TPR=0.7030  Prec=0.7416
  2. Pullover        TPR=0.8380  Prec=0.8136
  3. Coat            TPR=0.8420  Prec=0.8353
